# Final Class Document Modeling Review: JSON, MongoDB, and NoSQL Thinking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_15/final_class_document_modeling_review_notebook.ipynb)

This notebook reviews MongoDB and NoSQL ideas without requiring an Atlas login.

It uses Python dictionaries and lists because they look like JSON documents. The ideas transfer to MongoDB documents:

- flexible document shape
- embedding
- referencing
- access patterns
- aggregation-style summaries
- source-of-truth decisions
- vector similarity intuition

This notebook is for learning and review. It is not a replacement for running MongoDB Atlas.


## 1. JSON-Like Documents

MongoDB stores documents.

In Python, a document can be represented as a dictionary.

In JSON, the same idea uses objects, arrays, strings, numbers, booleans, and null values.


In [ ]:
# A simple document can contain nested data.
order_document = {
    "order_id": "ord_1001",
    "student": "Avery",
    "status": "paid",
    "items": [
        {"name": "burrito", "price": 8.50},
        {"name": "drink", "price": 2.00},
    ],
    "delivery": {
        "method": "pickup",
        "building": "Library Cafe",
    },
}

order_document


## 2. Embedding Review

Embedding means storing related data inside the same document.

Embedding can be a good choice when the application usually reads the data together.

Example: an order and its line items are often read together.


In [ ]:
# Embedded line items make this question easy:
# What is the total price of this one order?

total = 0
for item in order_document["items"]:
    total += item["price"]

print("Order:", order_document["order_id"])
print("Total price:", total)


## 3. Referencing Review

Referencing means storing an identifier that points to another document.

Referencing can be a good choice when:

- the related data changes independently
- the related data is shared by many records
- one source of truth matters

Example: many orders might reference the same student record.


In [ ]:
students = {
    "stu_1": {"student_id": "stu_1", "name": "Avery Rivera", "email": "avery@example.edu"},
    "stu_2": {"student_id": "stu_2", "name": "Jordan Lee", "email": "jordan@example.edu"},
}

orders = [
    {"order_id": "ord_1001", "student_id": "stu_1", "item": "burrito", "status": "paid"},
    {"order_id": "ord_1002", "student_id": "stu_2", "item": "salad", "status": "paid"},
    {"order_id": "ord_1003", "student_id": "stu_1", "item": "soup", "status": "pending"},
]

# Resolve the referenced student for each order.
for order in orders:
    student = students[order["student_id"]]
    print(order["order_id"], "belongs to", student["name"])


## 4. Access Patterns Decide The Shape

A common MongoDB modeling rule is:

> Model data based on how the application reads and writes it.

There is rarely only one correct answer.

Ask:

- What screen or query is most important?
- What data is read together?
- What data changes separately?
- What data must have one source of truth?


In [ ]:
# Access pattern: show each student's paid orders.
# This example uses referenced documents, so we combine data in application code.

for student_id, student in students.items():
    paid_orders = []
    for order in orders:
        if order["student_id"] == student_id and order["status"] == "paid":
            paid_orders.append(order["order_id"])

    print(student["name"], "paid orders:", paid_orders)


## 5. Aggregation-Style Thinking

MongoDB aggregation pipelines group, filter, reshape, and summarize documents.

This pure Python example reviews the same idea: count orders by status.


In [ ]:
from collections import Counter

status_counts = Counter()

for order in orders:
    status_counts[order["status"]] += 1

print("Order counts by status:")
for status, count in status_counts.items():
    print(status, count)


## 6. Source Of Truth Review

A source of truth is the system the application treats as official when copies disagree.

In distributed systems, logs, caches, events, and analytics records may be useful but incomplete.

The key question is not only where data exists. The key question is which copy the application is allowed to trust.


In [ ]:
official_orders = [
    {"order_id": "ord_1001", "payment_status": "paid"},
    {"order_id": "ord_1002", "payment_status": "paid"},
    {"order_id": "ord_1003", "payment_status": "paid"},
]

checkout_events = [
    {"event_type": "checkout_success", "order_id": "ord_1001"},
    {"event_type": "checkout_success", "order_id": "ord_1002"},
]

# Find paid orders that are missing checkout events.
orders_with_events = {event["order_id"] for event in checkout_events}

for order in official_orders:
    if order["payment_status"] == "paid" and order["order_id"] not in orders_with_events:
        print("Missing event for paid order:", order["order_id"])


## 7. Idempotent Retry Review

Idempotency means repeating the same logical operation does not create duplicate results.

This matters when a network request times out and the app retries.


In [ ]:
events_by_key = {}

# The idempotency key identifies the logical event.
new_event = {
    "idempotency_key": "checkout_success:ord_1003",
    "event_type": "checkout_success",
    "order_id": "ord_1003",
}

# First attempt inserts the event.
events_by_key.setdefault(new_event["idempotency_key"], new_event)

# Second attempt does not create a duplicate.
events_by_key.setdefault(new_event["idempotency_key"], new_event)

print("Number of stored events:", len(events_by_key))
print(list(events_by_key.values()))


## 8. Vector Similarity Intuition

Vector databases store items as lists of numbers called vectors.

A similarity search asks:

> Which stored item is most similar to this query vector?

Cosine similarity is useful because it compares direction, not just raw distance. That is helpful when vectors represent meaning.


In [ ]:
import math

# Tiny fake vectors for review only.
# Real embeddings are much longer and are produced by machine learning models.
query = [1.0, 0.8, 0.1]
concept_vectors = {
    "indexes": [0.9, 0.7, 0.1],
    "backups": [0.1, 0.2, 0.9],
    "document_modeling": [0.8, 0.9, 0.2],
}

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (length_a * length_b)

scores = []
for concept, vector in concept_vectors.items():
    scores.append((concept, cosine_similarity(query, vector)))

scores.sort(key=lambda item: item[1], reverse=True)

for concept, score in scores:
    print(concept, round(score, 3))


## Final Reflection

Use these prompts for your GitHub concept artifact or interview preparation:

- The NoSQL concept I can explain best is _____.
- The access pattern is _____.
- I would embed data when _____.
- I would reference data when _____.
- One tradeoff is _____.
